# Lab 42 (solution): Hardening the operations loop

Reference implementation. Three gaps in the [Lab 41](../../41-operating-the-loop/) loop, closed: the notifier becomes severity-aware and provider-shaped (Slack/PagerDuty/issue); the drift baseline re-records itself on promote so it tracks the current model; and a fixed **canary** set keeps the drift check and nightly job meaningful on a quiet day.

The hardened scripts (`notify.py`, `drift_check.py`, `run_loop.py`) and new tools (`record_baseline.py`, `canary.py`, `canary_queries.jsonl`) live in the [operating-the-loop](../../41-operating-the-loop/) toolkit; the workflows are updated in place.

## Step 0: Setup

In [ ]:
import json
import pathlib
import sys
# Lab 42 hardens the Lab 41 operating loop. The hardened scripts live in the operating-
# the-loop toolkit; point at it so we can import and exercise them.
loop = pathlib.Path.cwd().parent / "41-operating-the-loop"
sys.path.insert(0, str(loop))
print("hardening:", loop.name)

## Step 1: A real notifier (item 1)

Severity tiers + provider adapters.

In [ ]:
from notify import severity, format_alert, to_slack, to_pagerduty, to_github_issue, route_alert, post
# Item 1: a generic webhook is not on-call. Add severity (a dip warns, a collapse pages)
# and provider-shaped payloads, keeping the safe no-op default.
for v in [0.80, 0.74, 0.55]:
    print(f"value {v} vs 0.764 -> severity {severity(v, 0.764)}")

page = format_alert("judged_faithfulness", 0.55, 0.764, run_url="https://…/run/42")
print("\nSlack:    ", to_slack(page)["text"][:70], "…")
pd = to_pagerduty(page, "RK")
print("PagerDuty:", pd["event_action"], pd["payload"]["severity"])
print("Issue:    ", to_github_issue(page)["title"])
warn = format_alert("m", 0.74, 0.764)
print("\na WARN does not trigger PagerDuty:", to_pagerduty(warn, "RK")["event_action"])
print(post(to_slack(page), url=None))   # no endpoint configured -> no-op

## Step 2: A self-updating drift baseline (item 2)

Promote re-records it; the check reads the current one.

In [ ]:
from record_baseline import compute_baseline
from drift_check import load_baseline, drift_status
# Item 2: the drift baseline must track the CURRENT model. The promote phase re-records
# it; the drift check reads the recorded file (falling back to a constant before the
# first record).
new_baseline = compute_baseline([0.74, 0.71, 0.76, 0.73, 0.72])   # a retrained model's confidences
print("recomputed baseline:", new_baseline)
m, s = load_baseline()   # reads confidence_baseline.json if present, else the constant
print(f"drift check will compare against mean={m}, std={s}")
# A window healthy under the OLD baseline can read as drift under a tighter NEW one - which
# is exactly why you re-record: the band has to match the model it is judging.
print("window 0.70 status vs recorded baseline:", drift_status([0.70,0.69,0.71], m, s)["status"])

## Step 3: Canaries (item 4)

A heartbeat for quiet days; a flipped route is a hard break.

In [ ]:
from canary import load_canaries, augment_window, canary_routing_failures
# Item 4: volume-based signals go dark on a quiet day. A fixed canary set is a heartbeat.
cans = load_canaries()
print(f"{len(cans)} canaries across {len({c['route'] for c in cans})} routes")
# always include them in the drift window
print("augmented window size (1 live + canaries):", len(augment_window([{"query":"a live query"}], cans)))
# a flipped canary route is a hard break, independent of any threshold
preds = [c["route"] for c in cans]
preds[0] = "parametric"
fails = canary_routing_failures(preds, cans)
print("simulated route flip -> failures:", fails)
print("In CI, canary.py exits non-zero on any such failure (a loud, threshold-free signal).")

## Step 4: The hardened cadence

In [ ]:
# The hardened cadence (workflows updated in place):
#   rag-drift-check.yml        + canary routing check (hard) + --canaries in the window
#   rag-faithfulness-nightly   + severity-aware notify routed to Slack/PagerDuty
#   rag-maintenance-loop.yml   promote now re-records BOTH baselines (gate + drift)
print("Drift check carries canaries and fails on a flipped route.")
print("Nightly pages or warns by severity, to a real channel.")
print("Promote refreshes the gate thresholds (Lab 38) AND the drift baseline (Lab 42).")

## Step 5: What hardening buys

In [ ]:
# What 'hardening' means here: the loop no longer depends on a human noticing.
#  - alerts reach on-call with a severity, not a buried summary line;
#  - the drift baseline can't go stale because promote rewrites it;
#  - a quiet traffic day can't hide a break, because canaries always run.
print("A maintained system pages the right person at the right severity, compares against")
print("a current baseline, and keeps a heartbeat even when traffic is silent.")

## What you built

The production-hardening pass on the maintenance loop: `notify.py` now decides severity (warn vs page) and shapes payloads for Slack, PagerDuty, or a GitHub issue (safe no-op until you configure an endpoint); `record_baseline.py` recomputes the drift baseline and is called by the promote phase, so `drift_check.py` always compares against the current model; and `canary.py` + `canary_queries.jsonl` give the drift check and nightly job a fixed heartbeat, with a flipped canary route failing loudly regardless of traffic volume.

**Where this simplifies:** the provider payloads are minimal (no retries, no rate-limit or dedup/cooldown — add those before high volume); the page/warn split is a single fraction (`PAGE_FRACTION`) you tune to your tolerance; the canary set is ten queries (grow it to cover the failure modes you actually fear); the drift baseline reference is the prototype set (use a held-out clean sample if you have one).

Next: [Lab 43](../../43-annotator-drift/) hardens the *evaluation* side — catching drift in the annotators themselves, not just the model.